In [1]:
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
train_dir = "dataset/train"
val_dir   = "dataset/val"

print("Train classes:", sorted(os.listdir(train_dir)))
print("Val   classes:", sorted(os.listdir(val_dir)))

Train classes: ['Diagram', 'Mixed', 'Text']
Val   classes: ['Diagram', 'Mixed', 'Text']


In [3]:
# ── Model ──────────────────────────────────────────────────────────────────
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPool2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPool2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPool2D(2, 2),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.4),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(3, activation="softmax"),
])

c:\Users\anura\Downloads\AI_LEARN\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
# ── Compile ────────────────────────────────────────────────────────────────
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [5]:
# ── Data generators ────────────────────────────────────────────────────────
# NOTE: val_datagen uses rescale ONLY — no augmentation on validation data
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True,
    zoom_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)

In [6]:
# ── Train generator ────────────────────────────────────────────────────────
# CRITICAL: classes= forces label order to match app.py's content_map
#   content_map = {0: 'Text', 1: 'Diagram', 2: 'Mixed'}
# Without this, flow_from_directory sorts alphabetically:
#   Diagram=0, Mixed=1, Text=2  ← WRONG for app.py
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    classes=["Text", "Diagram", "Mixed"],  # Text=0, Diagram=1, Mixed=2
)

print("class_indices:", train_gen.class_indices)  # verify order

Found 600 images belonging to 3 classes.
class_indices: {'Text': 0, 'Diagram': 1, 'Mixed': 2}


In [7]:
# ── Val generator ──────────────────────────────────────────────────────────
val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    classes=["Text", "Diagram", "Mixed"],  # must match train_gen
)

print("class_indices:", val_gen.class_indices)  # verify order

Found 600 images belonging to 3 classes.
class_indices: {'Text': 0, 'Diagram': 1, 'Mixed': 2}


In [8]:
# ── Callbacks ──────────────────────────────────────────────────────────────
os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath="models/cnn_best.h5",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

In [9]:
# ── Train ──────────────────────────────────────────────────────────────────
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks,
)

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4856 - loss: 5.0280
Epoch 1: val_accuracy improved from None to 0.33333, saving model to models/cnn_best.h5



Epoch 1: finished saving model to models/cnn_best.h5
19/19 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.5817 - loss: 4.4799 - val_accuracy: 0.3333 - val_loss: 1.4939
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 930ms/step - accuracy: 0.6524 - loss: 2.1290
Epoch 2: val_accuracy did not improve from 0.33333
19/19 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.6767 - loss: 1.7151 - val_accuracy: 0.3333 - val_loss: 13.7956
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 958ms/step - accuracy: 0.7606 - loss: 1.1987
Epoch 3: val_accuracy did not improve from 0.33333
19/19 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.7733 - loss: 1.0356 - val_accuracy: 0.3333 - val_loss: 26.3356
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 960ms/step - accuracy: 0.8458 - loss: 0.6625
Epoch 4: val_accuracy did not improve from 0.33333
19/19 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.8567 - loss: 0.5992 - val_accuracy: 0.3333 - val_loss: 27.4934
Epoch 4: early stopping
Restoring model weights from the end of the best

In [10]:
# ── Summary & save ─────────────────────────────────────────────────────────
model.summary()

# Save the final weights (ModelCheckpoint already saved the best epoch above)
model.save("models/cnn_classifier.h5")
print("\n[DONE] CNN model saved → models/cnn_classifier.h5")
print("       Best epoch   → models/cnn_best.h5")
print("       Class order  → Text=0, Diagram=1, Mixed=2")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,941,067 (37.92 MB)

 Trainable params: 3,313,539 (12.64 MB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 6,627,080 (25.28 MB)


[DONE] CNN model saved → models/cnn_classifier.h5
       Best epoch   → models/cnn_best.h5
       Class order  → Text=0, Diagram=1, Mixed=2
